# Khmer News Classification Using Traditional Machine Learning Algorithms
## A Reproducible Experimental Workflow on the KhNeAC Dataset

| | |
|---|---|
| **Course** | Fundamentals of Machine Learning |
| **Authors** | *[Author Name 1, Author Name 2, ...]* |
| **Affiliation** | *Data Science Program, [University]* |
| **Date** | 2026-11-10 |
| **License** | MIT (code) — fork and adapt freely; cite the dataset (below) |

**Abstract.** This notebook implements a complete, reproducible machine learning workflow for Khmer-language news classification. It compares four traditional algorithms (Multinomial Naive Bayes, Logistic Regression, Linear SVM, Random Forest) under two feature representations (Bag of Words, TF-IDF — word- and character-level) on the KhNeAC dataset, which provides 7,344 labeled Khmer news articles in six preprocessing configurations (raw, clean, normalized, compound-segmented, morpheme-segmented, and KCC-segmented). Models are tuned on the official validation split and evaluated on the held-out test split with accuracy, macro-averaged F1, per-class reports, and confusion matrices. The notebook is organized to be forked and reused: all design choices are centralized in a configuration cell, and each stage is documented with its academic rationale.

**Keywords.** Khmer NLP; text classification; TF-IDF; word segmentation; model comparison; reproducibility

**Data citation.** CADT Inclusive Digital Innovation Research Lab (IDRI), *KhNeAC: Khmer News Article Classification Dataset*, Hugging Face: https://huggingface.co/datasets/sopagnaheang/KhNeAC (derived from `CADT-IDRI/Khmer_News_classification`).

> **How to run.** Runtime ≈ 10–20 min CPU (free Colab tier is sufficient). Run cells top-to-bottom. Section 8 (segmentation study) retrains across all six dataset configurations and is the most expensive part.

## 1. Environment Setup

The workflow requires `datasets` (Hugging Face) for data loading and `scikit-learn ≥ 1.3` (the `dual="auto"` option of `LinearSVC`). Google Colab satisfies all other dependencies by default.

**Reproducibility protocol.** A fixed random seed (`SEED = 42`) is applied to every stochastic component (train/validation subsampling, Random Forest). Linear models and Naive Bayes are deterministic given the same data.

In [ ]:
# Core dependency (uncomment if missing)
%pip install -q datasets

In [ ]:
# Imports, global configuration, and reproducibility
import os, re, random, unicodedata, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from datasets import load_dataset
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             classification_report, confusion_matrix)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", font_scale=0.9)
%matplotlib inline

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# ----------------------------------------------------------------------------
# GLOBAL CONFIGURATION — centralize every design choice here so the notebook
# can be forked and adapted to another dataset or another set of experiments.
# ----------------------------------------------------------------------------
CONFIG = {
    # Data
    "dataset_name":   "sopagnaheang/KhNeAC",
    "configs":        ["raw", "clean", "normalize", "com_seg", "mor_seg", "kcc_seg"],
    "segmented":      {"com_seg", "mor_seg", "kcc_seg"},   # configs with word boundaries
    "main_config":    "normalize",     # config used for the main pipeline (§2–§7)
    "use_title":      True,            # concatenate title + content as the document
    "sample_size":    None,            # e.g. 1000 for a quick smoke test; None = full data
    # Features
    "max_features":   20000,           # vocabulary cap for all vectorizers
    "min_df":         2,               # ignore terms in < 2 documents
    "word_ngrams":    (1, 2),
    "char_ngrams":    (2, 4),
    # Models
    "seed":           SEED,
    "n_jobs":         -1,
}

print("scikit-learn, datasets, pandas, matplotlib imported — ready.")

## 2. Data Loading

**Why six configurations?** Khmer is written without whitespace word boundaries, so the dataset authors released the same 7,344 articles under six preprocessing variants (KhNeAC, 2026). This turns segmentation itself into a *controlled experimental factor*: any performance difference between configurations is attributable to the text-processing pipeline, not to the data.

| Config | Description | Word boundaries? |
|---|---|---|
| `raw` | Original scraped text | No |
| `clean` *(default)* | Whitespace trimmed, invisible Unicode removed | No |
| `normalize` | Cleaned + Khmer text normalization | No |
| `com_seg` | Normalized + **compound-word** segmentation | Yes |
| `mor_seg` | Normalized + **morpheme** segmentation | Yes |
| `kcc_seg` | Normalized + **KCC** (Khmer Character Cluster) segmentation | Yes |

**Official splits.** Train 5,140 · Validation 661 · Test 1,543 (fixed; we never re-split, so validation-based tuning cannot leak into test).

**Data fields.** `index_id` (int), `title` (str), `content` (str), `label` (6 classes: 0 `economic`, 1 `entertainment`, 2 `life`, 3 `politic`, 4 `sport`, 5 `technology`).

**Loading strategy.** The *main* configuration is loaded eagerly; all six are accessible through a cached helper (`get_splits`) used later by the segmentation study (§8), so nothing is loaded twice.

In [ ]:
LABELS = ["economic", "entertainment", "life", "politic", "sport", "technology"]

def get_splits(config: str) -> dict:
    '''Load one KhNeAC configuration and return train/validation/test DataFrames.

    Each DataFrame contains a single ready-to-use text column (title+content,
    optionally preprocessed later in §4) and integer labels. Results are cached.
    '''
    if config in get_splits._cache:
        return get_splits._cache[config]

    ds = load_dataset(CONFIG["dataset_name"], config)
    names = ds["train"].features["label"].names  # authoritative label order

    out = {}
    for split in ["train", "validation", "test"]:
        df = ds[split].to_pandas()
        text = df["title"].fillna("") + " " + df["content"].fillna("") \
            if CONFIG["use_title"] else df["content"].fillna("")
        out[split] = pd.DataFrame({"text": text, "label": df["label"].astype(int)})

    if CONFIG["sample_size"]:  # optional stratified smoke-test subsampling
        for split in list(out):
            n = min(CONFIG["sample_size"], len(out[split]))
            idx, _ = train_test_split(out[split].index, train_size=n,
                                      stratify=out[split]["label"], random_state=SEED)
            out[split] = out[split].loc[idx].reset_index(drop=True)

    out["label_names"] = names
    get_splits._cache[config] = out
    print(f"[{config}] train={len(out['train'])}  val={len(out['validation'])}  "
          f"test={len(out['test'])}  labels={names}")
    return out

get_splits._cache = {}

# Load the main configuration only (others are loaded lazily in §8)
splits = get_splits(CONFIG["main_config"])
train_df, val_df, test_df = splits["train"], splits["validation"], splits["test"]
train_df.head(2)

## 3. Exploratory Data Analysis

EDA precedes modeling for two evidentiary reasons. First, **class balance**: if categories are imbalanced, plain accuracy over-represents majority classes, which independently justifies macro-averaged F1 as the primary metric. Second, **document length and vocabulary size**: these determine sensible values for `min_df`, `max_features`, and n-gram ranges — parameters that would otherwise be arbitrary.

In [ ]:
# 3.1 Class distribution across the three official splits
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4), sharey=True)
for ax, (name, df) in zip(axes, [("Train", train_df), ("Validation", val_df), ("Test", test_df)]):
    counts = df["label"].value_counts().sort_index()
    ax.bar([LABELS[i] for i in counts.index], counts.values, color="#3b6ea5")
    ax.set_title(f"{name} (n={len(df)})"); ax.tick_params(axis="x", rotation=45)
axes[0].set_ylabel("Articles")
plt.suptitle("Class distribution — KhNeAC main config", y=1.04)
plt.tight_layout(); plt.show()

print(train_df["label"].value_counts().sort_index().rename(
    index={i: l for i, l in enumerate(LABELS)}))

In [ ]:
# 3.2 Document length and vocabulary statistics
stats = []
for name, df in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    char_len = df["text"].str.len()
    tok_len  = df["text"].str.split().str.len()   # whitespace tokens (meaningful only if segmented)
    vocab    = set(w for doc in df["text"] for w in doc.split())
    stats.append({"split": name, "docs": len(df),
                  "chars: mean": round(char_len.mean()), "chars: max": int(char_len.max()),
                  "ws-tokens: mean": round(tok_len.mean()),
                  "ws-tokens: max": int(tok_len.max()),
                  "ws-vocab": len(vocab)})
pd.DataFrame(stats)

## 4. Data Preprocessing

Khmer web text contains artifacts that add noise dimensions to the feature space without adding semantic value. The function below implements a **conservative, language-aware cleaning policy**; every decision is documented:

| Artifact | Treatment | Rationale |
|---|---|---|
| Unicode incompatibility (e.g., precomposed vs. decomposed clusters) | NFC normalization | Makes identical strings identical for the vectorizer |
| URLs, e-mails, HTML tags | Removed | Source metadata, not news content; inflate vocabulary with near-unique tokens |
| Zero-width characters (`U+200B`–`U+200D`, BOM) | Removed | Invisible; match the dataset's own `clean` policy |
| Sentence-final Khmer punctuation (`។` `៕` `៖` `៘`) | Replaced by space | Acts as a pseudo word boundary; safest consistent separator |
| Other punctuation / symbols | Replaced by space | Keeps letters of all scripts and **Khmer combining marks** (vowel signs, COENG `្` U+17D2 — `ា–៓`) |
| Latin + Khmer digits | Replaced by `<num>` token | Dates/amounts generalize as a type, not as distinct strings |
| Repeated whitespace | Collapsed | Canonical form |

Two deliberate **non-decisions** are worth stating: (i) we do *not* lowercase-strip Khmer (Khmer has no case) — only Latin text is lowercased; (ii) we do *not* remove stop-words at this stage — TF-IDF weighting already attenuates high-frequency function words, and premature removal risks deleting discriminative short tokens.

The transformation is **idempotent and cheap**, so it is applied uniformly to all configurations (including already-normalized ones) — on `clean`/`normalize` it is nearly a no-op, which keeps the experiment matrix comparable.

In [ ]:
ZW_CHARS = ["\u200b", "\u200c", "\u200d", "\ufeff"]          # ZWSP, ZWNJ, ZWJ, BOM
KHMER_SENT_PUNCT = "។៕៖៘"

def preprocess_khmer(text: str) -> str:
    '''Conservative Khmer-aware text cleaning (see §4 table for the rationale).'''
    text = unicodedata.normalize("NFC", str(text))
    text = text.lower()                                        # Latin only; Khmer has no case
    text = re.sub(r"<[^>]+>", " ", text)                       # HTML tags
    text = re.sub(r"(https?://\S+|www\.\S+)", " ", text)       # URLs
    text = re.sub(r"\S+@\S+\.\S+", " ", text)                  # e-mails
    for ch in ZW_CHARS:
        text = text.replace(ch, "")                            # invisible characters
    text = re.sub(f"[{KHMER_SENT_PUNCT}]", " ", text)          # Khmer sentence punctuation
    # Remove remaining symbols, but PRESERVE Khmer combining marks (category Mn):
    # vowel signs and COENG (U+17D2, subscript consonants) are integral parts of
    # Khmer words -- Python's \w does NOT match them, so they are kept explicitly.
    text = re.sub(r"[^\w\s\u17B6-\u17D3\u17DD]", " ", text)
    text = re.sub(r"\d+", " <num> ", text)                     # Arabic + Khmer digits (Nd)
    return re.sub(r"\s+", " ", text).strip()

# Apply once, cache on the DataFrames
for df in (train_df, val_df, test_df):
    df["text_clean"] = df["text"].map(preprocess_khmer)

In [ ]:
# 4.1 Before/after demonstration on real articles
sample = train_df.sample(2, random_state=SEED)
for _, row in sample.iterrows():
    print("RAW   :", row["text"][:160].replace("\n", " "), "…")
    print("CLEAN :", row["text_clean"][:160], "…\n" + "-" * 100)

diff = (train_df["text"].str.len() - train_df["text_clean"].str.len())
print(f"Characters removed: mean {diff.mean():.1f} / doc ({diff.mean() / train_df['text'].str.len().mean():.1%})")

## 5. Feature Extraction

Text must be converted into numerical vectors before any traditional algorithm can operate. We use two **term-frequency-based** representations:

**Bag of Words (BoW)** counts raw term occurrences:
$\mathbf{x}_{d,t} = \mathrm{tf}(t, d)$

**TF-IDF** re-weights counts by inverse document frequency, down-weighting terms that occur across many documents (function words, editorial boilerplate) and up-weighting discriminative ones:
$\mathbf{x}_{d,t} = \mathrm{tf}(t, d)\cdot \log\!\frac{N}{1+\mathrm{df}(t)}$ (with sub-linear TF scaling)

**The word-vs-character decision is the central design point of this project.**

| Representation | Analyzer | Requires segmentation? | Used for | Why |
|---|---|---|---|---|
| BoW / TF-IDF (word) | word n-grams `(1,2)` | **Yes** | `com_seg`, `mor_seg`, `kcc_seg` | Tokens are real words only where segmentation inserted spaces; on unsegmented Khmer a "whitespace token" would be a whole clause |
| TF-IDF (char) | `char_wb` n-grams `(2,4)` | **No** | `raw`, `clean`, `normalize` | Character n-grams learn recurring subword units (prefixes, clusters, common fragments) and are robust to imperfect segmentation |

All vectorizers share `max_features=20000`, `min_df=2` (noise/terms seen once are excluded), and `sublinear_tf` for TF-IDF. Hyper-parameters were chosen a priori and are exposed in `CONFIG` for ablation.

In [ ]:
def make_vectorizer(kind: str):
    '''Feature-extraction factory.

    kind = 'bow'      → word-level Bag of Words          (needs segmented text)
    kind = 'word_tf'  → word-level TF-IDF               (needs segmented text)
    kind = 'char_tf'  → character n-gram TF-IDF         (works without segmentation)
    '''
    if kind == "bow":
        return CountVectorizer(max_features=CONFIG["max_features"], min_df=CONFIG["min_df"],
                               ngram_range=(1, 1), lowercase=False,
                               token_pattern=r"(?u)\b\w+\b")
    if kind == "word_tf":
        return TfidfVectorizer(max_features=CONFIG["max_features"], min_df=CONFIG["min_df"],
                               ngram_range=CONFIG["word_ngrams"], sublinear_tf=True,
                               lowercase=False, token_pattern=r"(?u)\b\w+\b")
    if kind == "char_tf":
        return TfidfVectorizer(max_features=CONFIG["max_features"], min_df=CONFIG["min_df"],
                               analyzer="char_wb", ngram_range=CONFIG["char_ngrams"],
                               sublinear_tf=True, lowercase=False)
    raise ValueError(kind)

# Dimensionality preview on the main configuration.
# Word features are only meaningful on segmented configs; char features always work.
kind = "word_tf" if CONFIG["main_config"] in CONFIG["segmented"] else "char_tf"
V = make_vectorizer(kind).fit_transform(train_df["text_clean"])
sparsity = 1 - V.nnz / (V.shape[0] * V.shape[1])
print(f"{kind} on '{CONFIG['main_config']}': matrix {V.shape}, sparsity {sparsity:.1%}")

## 6. Model Training

**Candidate algorithms** (all *traditional* ML — no neural networks — per the project scope):

| Model | Family | Role |
|---|---|---|
| Multinomial Naive Bayes | Generative, bag-of-words | Fast probabilistic baseline [5, 7] |
| Logistic Regression | Discriminative linear | Well-calibrated linear baseline |
| Linear SVM | Discriminative linear (max-margin) | Historically strongest on high-dim sparse text [6] |
| Random Forest | Non-linear ensemble | Tests whether feature interactions add value |

**Experimental protocol (leakage-free).**
1. For each experiment, a small grid is tuned **on the validation split only** (macro F1).
2. The best hyper-parameter is kept; the model fitted on train is used on test **once**.
3. The test set is never consulted for tuning or early stopping — reported test numbers are unbiased.

Each experiment is a sklearn `Pipeline(vectorizer → classifier)`, which prevents any feature-fitting leakage and makes the whole experiment serializable as one object (§9).

In [ ]:
# 6.1 Experiment registry
# feature kind: 'word_tf'/'bow' require segmented configs; 'char_tf' works on any config.
EXPERIMENTS = [
    # --- core comparison (proposal, Table 3): run on the main config ---
    {"id": "E1", "config": CONFIG["main_config"], "features": "bow",     "clf": "nb",
     "grid": {"clf__alpha": [0.1, 0.5, 1.0]}},
    {"id": "E2", "config": CONFIG["main_config"], "features": "word_tf", "clf": "nb",
     "grid": {"clf__alpha": [0.1, 0.5, 1.0]}},
    {"id": "E3", "config": CONFIG["main_config"], "features": "word_tf", "clf": "lr",
     "grid": {"clf__C": [0.5, 1.0, 2.0]}},
    {"id": "E4", "config": CONFIG["main_config"], "features": "word_tf", "clf": "svc",
     "grid": {"clf__C": [0.5, 1.0, 2.0]}},
    {"id": "E5", "config": CONFIG["main_config"], "features": "word_tf", "clf": "rf",
     "grid": {"clf__n_estimators": [100, 200]}},
    # --- extensions (proposal §9, H1/H5) ---
    {"id": "E6", "config": CONFIG["main_config"], "features": "char_tf", "clf": "svc",
     "grid": {"clf__C": [0.5, 1.0, 2.0]}},
    {"id": "E7", "config": CONFIG["main_config"], "features": "bow",     "clf": "lr",
     "grid": {"clf__C": [0.5, 1.0, 2.0]}},
]

def make_model(clf: str):
    '''Classifier factory (all stochastic parts are seeded).'''
    return {
        "nb":  MultinomialNB(),
        "lr":  LogisticRegression(max_iter=1000, random_state=SEED),
        "svc": LinearSVC(random_state=SEED, dual="auto"),
        "rf":  RandomForestClassifier(n_estimators=150, max_features="sqrt",
                                      n_jobs=CONFIG["n_jobs"], random_state=SEED),
    }[clf]

def make_pipeline(features: str, clf: str) -> Pipeline:
    return Pipeline([("vec", make_vectorizer(features)), ("clf", make_model(clf))])

print(f"{len(EXPERIMENTS)} experiments registered.")

In [ ]:
# 6.2 Tuning + fitting engine
def run_experiment(exp: dict) -> dict:
    '''Tune on validation (macro F1), then evaluate the selected model on test.

    Returns one results row plus the fitted pipeline and test predictions.
    The test split is touched exactly once, after model selection.
    '''
    sp = get_splits(exp["config"])
    X_tr, y_tr = sp["train"]["text_clean"], sp["train"]["label"]
    X_va, y_va = sp["validation"]["text_clean"], sp["validation"]["label"]
    X_te, y_te = sp["test"]["text_clean"], sp["test"]["label"]

    pipe = make_pipeline(exp["features"], exp["clf"])

    # -- validation-based grid search (macro F1) --
    best_f1, best_params, best_val_f1_per_param = -1.0, None, {}
    for params in [dict(zip(g, v)) for g, vals in exp["grid"].items() for v in vals]:
        pipe.set_params(**params).fit(X_tr, y_tr)
        f1v = f1_score(y_va, pipe.predict(X_va), average="macro")
        best_val_f1_per_param[str(params)] = round(f1v, 4)
        if f1v > best_f1:
            best_f1, best_params = f1v, params
    pipe.set_params(**best_params).fit(X_tr, y_tr)     # refit with the selected params

    # -- one-shot test evaluation --
    y_pred = pipe.predict(X_te)
    return {"experiment": exp["id"], "config": exp["config"], "features": exp["features"],
            "clf": exp["clf"], "best_params": best_params, "val_macro_f1": round(best_f1, 4),
            "test_accuracy": round(accuracy_score(y_te, y_pred), 4),
            "test_macro_f1": round(f1_score(y_te, y_pred, average="macro"), 4),
            "test_weighted_f1": round(f1_score(y_te, y_pred, average="weighted"), 4),
            "pipeline": pipe, "y_test": y_te, "y_pred": y_pred}

print("run_experiment() defined.")

## 7. Evaluation

Models are compared with **accuracy**, **macro-F1** (primary — each of the six categories contributes equally, so majority classes cannot mask minority-class failures), **weighted-F1**, per-class precision/recall/F1, and confusion matrices. Per-class reports identify which categories are easy (high intra-class vocabulary distinctiveness) and which are systematically confused (shared vocabulary across related categories).

In [ ]:
# 7.1 Run the full experiment matrix (≈ 10 min on CPU; E5/RF is the slow one)
results, fitted = [], {}
for exp in EXPERIMENTS:
    row = run_experiment(exp)
    fitted[exp["id"]] = row
    results.append({k: v for k, v in row.items() if k not in ("pipeline", "y_test", "y_pred")})
    print(f"{row['experiment']}  features={row['features']:<8} clf={row['clf']:<4} "
          f"valF1={row['val_macro_f1']:.3f}  testAcc={row['test_accuracy']:.3f}  "
          f"testMacroF1={row['test_macro_f1']:.3f}")

results_df = (pd.DataFrame(results)
              .drop_duplicates(subset=["experiment"], keep="first")
              .sort_values("test_macro_f1", ascending=False).reset_index(drop=True))
results_df

In [ ]:
# 7.2 Best model: classification report + confusion matrix on the test split
best_id = results_df.iloc[0]["experiment"]
best = fitted[best_id]
names = get_splits(best["config"])["label_names"]

print(f"Best experiment: {best_id}  ({best['features']} + {best['clf']})")
print(classification_report(best["y_test"], best["y_pred"], target_names=names, digits=3))

cm = confusion_matrix(best["y_test"], best["y_pred"])
plt.figure(figsize=(6.5, 5.2))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=names, yticklabels=names, cbar=False)
plt.title(f"Confusion matrix — {best_id} (test)")
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.xticks(rotation=45)
plt.tight_layout(); plt.show()

In [ ]:
# 7.3 Per-class F1 of the best model (which categories are easy / hard?)
rep = classification_report(best["y_test"], best["y_pred"], target_names=names,
                            output_dict=True)
per_class = (pd.DataFrame(rep).T
             .loc[names, ["precision", "recall", "f1-score"]].round(3))
per_class["f1-score"].sort_values().plot.barh(color="#3b6ea5", figsize=(7, 3.4))
plt.xlabel("Test F1"); plt.title(f"Per-class F1 — {best_id}")
plt.tight_layout(); plt.show()
per_class

## 8. Insights & Discussion

Results are interpreted against the five hypotheses registered in the project proposal (stated *a priori*, tested — not assumed):

- **H1.** TF-IDF outperforms BoW (E1 vs E2).
- **H2.** Word segmentation improves classification (segmented vs unsegmented configs, §8.2).
- **H3.** Linear SVM outperforms Naive Bayes in high-dimensional spaces (E2 vs E4).
- **H4.** Politics and Economics are the most confused pair (shared vocabulary, §8.4).
- **H5.** Character-level TF-IDF mitigates imperfect segmentation (E6 vs E4).

In [ ]:
# 8.1 Overall comparison of experiments (test macro F1)
ax = results_df.plot.barh(x="experiment", y="test_macro_f1", legend=False,
                          figsize=(7, 3.6), color="#3b6ea5")
ax.set_xlabel("Test macro-F1"); ax.set_title("Experiment comparison (test split)")
for i, v in enumerate(results_df["test_macro_f1"]):
    ax.text(v + 0.003, i, f"{v:.3f}", va="center", fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# 8.2 Segmentation study (H2): same model across all six dataset configurations
#     word TF-IDF on every config  +  char TF-IDF on unsegmented configs
seg_rows = []
for cfg in CONFIG["configs"]:
    exp = {"id": f"SEG-{cfg}", "config": cfg, "features": "word_tf", "clf": "svc",
           "grid": {"clf__C": [1.0]}}
    r = run_experiment(exp)
    seg_rows.append({"config": cfg, "feature": "word_tf", "test_macro_f1": r["test_macro_f1"]})
    if cfg not in CONFIG["segmented"]:                     # char features are segmentation-free
        exp["features"] = "char_tf"; exp["id"] = f"SEG-{cfg}-char"
        r2 = run_experiment(exp)
        seg_rows.append({"config": cfg, "feature": "char_tf", "test_macro_f1": r2["test_macro_f1"]})

seg_df = pd.DataFrame(seg_rows)
plt.figure(figsize=(8, 4))
sns.barplot(data=seg_df, x="config", y="test_macro_f1", hue="feature", palette="Blues")
plt.title("Does segmentation help? (Linear SVM, test macro-F1)")
plt.ylabel("Test macro-F1"); plt.xlabel("Dataset configuration")
plt.tight_layout(); plt.show()
seg_df

In [ ]:
# 8.3 Most informative features per class (best Linear SVM, §7.2)
vec, clf = best["pipeline"].named_steps["vec"], best["pipeline"].named_steps["clf"]
feat_names = vec.get_feature_names_out()
for i, cls in enumerate(clf.classes_):
    top = feat_names[np.argsort(clf.coef_[i])[::-1][:12]]
    print(f"{names[cls]:<14} → {'  '.join(top)}")
print("\n(Tokens reflect the text configuration — words on segmented configs, "
      "subword fragments on char models.)")

In [ ]:
# 8.4 Most-confused category pairs (H4) — off-diagonal mass of the best model's CM
cm_df = pd.DataFrame(cm, index=names, columns=names)
pairs = (cm_df.stack().rename("count").reset_index()
         .rename(columns={"level_0": "actual", "level_1": "predicted"}))
pairs = pairs[pairs["actual"] != pairs["predicted"]].sort_values("count", ascending=False).head(8)
pairs["share_of_actual"] = (pairs["count"] /
                            cm_df.sum(axis=1).reindex(pairs["actual"]).values * 100).round(1)
pairs.reset_index(drop=True)

**Writing the discussion (replace this cell after running).** State each hypothesis as *supported / partially supported / not supported*, in hedged academic language, and connect numbers to causes. Useful sentence frames:

- *"TF-IDF outperformed BoW (macro-F1 Δ = …), which suggests that down-weighting high-frequency function words reduces noise in the Khmer feature space."*
- *"Segmented configurations outperformed `normalize` by … macro-F1 points, indicating that explicit word boundaries yield more informative lexical units."*
- *"The Politics↔Economics confusion (…% of Politics articles) may be attributed to overlapping vocabulary in trade and governance reporting."*
- *"Character-level features narrowed the gap for unsegmented text, consistent with their robustness to segmentation errors."*

| Model | Accuracy | Macro-F1 |
|---|---|---|
| Naive Bayes | *to be reported* | *to be reported* |
| Logistic Regression | *to be reported* | *to be reported* |
| Linear SVM | *to be reported* | *to be reported* |
| Random Forest | *to be reported* | *to be reported* |

## 9. Persistence & Reproducibility

The winning pipeline (vectorizer **and** classifier as one object) is serialized with `joblib`, together with the full results table — a submission artifact for grading and a starting point for future work (e.g., loading the pipeline to classify new articles with a single `.predict()` call).

In [ ]:
os.makedirs("artifacts", exist_ok=True)
joblib.dump(best["pipeline"], f"artifacts/khmer_news_{best_id}_{best['features']}_{best['clf']}.joblib")
results_df.to_csv("artifacts/results.csv", index=False)

import sklearn, matplotlib, datasets as hf
{"timestamp": pd.Timestamp.now().isoformat(),
 "python_env": {"scikit-learn": sklearn.__version__, "datasets": hf.__version__,
                "pandas": pd.__version__, "numpy": np.__version__,
                "matplotlib": matplotlib.__version__},
 "config": {k: v for k, v in CONFIG.items() if k != "segmented"},
 "best_experiment": best_id}

## 10. Adapting This Notebook (Fork Guide)

The notebook is deliberately configuration-driven — most adaptations require editing **only** the `CONFIG` cell and/or the `EXPERIMENTS` list:

| Goal | Change |
|---|---|
| Quick smoke test | `CONFIG["sample_size"] = 1000` |
| Use a different main config | `CONFIG["main_config"] = "kcc_seg"` |
| Ignore titles | `CONFIG["use_title"] = False` |
| Larger vocabulary | `CONFIG["max_features"] = 50000` |
| Test another algorithm | Add to `make_model()` + append an entry to `EXPERIMENTS` |
| Different n-gram ranges | `CONFIG["word_ngrams"]`, `CONFIG["char_ngrams"]` |
| **Different dataset** | Point `CONFIG["dataset_name"]` to any HF text-classification dataset with `text`-like and `label` columns; adjust the column names in `get_splits()` |

**Extension ideas** (ordered by effort): Chi² feature selection; calibrated SVM (`CalibratedClassifierCV`) for probabilities; error analysis on the longest documents; contrast fine-tuned Khmer BERT against this traditional baseline; deploy the serialized pipeline behind a small FastAPI service.

**Compute notes.** E5 (Random Forest) dominates runtime; reduce `n_estimators` or restrict to `max_features="sqrt"` (already set) for faster iterations. The §8.2 study fits 6–9 additional SVM pipelines and typically takes 2–4 minutes.

## References

1. CADT Inclusive Digital Innovation Research Lab (IDRI). *KhNeAC: Khmer News Article Classification Dataset*. Hugging Face. https://huggingface.co/datasets/sopagnaheang/KhNeAC (accessed 2026-11-10).
2. CADT IDRI. *Khmer News Classification (unsegmented)*. Hugging Face. https://huggingface.co/datasets/CADT-IDRI/Khmer_News_classification (accessed 2026-11-10).
3. Pedregosa, F., et al. (2011). Scikit-learn: Machine Learning in Python. *Journal of Machine Learning Research*, 12, 2825–2830.
4. Manning, C. D., Raghavan, P., & Schütze, H. (2008). *Introduction to Information Retrieval*. Cambridge University Press.
5. McCallum, A., & Nigam, K. (1998). A comparison of event models for Naive Bayes text classification. *AAAI-98 Workshop on Learning for Text Categorization*, 41–48.
6. Joachims, T. (1998). Text categorization with support vector machines: Learning with many relevant features. *ECML-98*, 137–142.
7. Zhang, H. (2004). The optimality of Naive Bayes. *FLAIRS-17*.
8. *Pretrained models and evaluation data for the Khmer language*. COLING 2022.
9. *Multi-class text classification on Khmer news using ensemble method in machine learning algorithms*. AIP Conference Proceedings, 2023.

---
*Notebook version 1.0 (2026-11-10). Distributed under the MIT License. If you fork this notebook, please retain the data citation above.*